# 🏗️ Object-Oriented Programming in Python — Classes That Power ML Libraries

> **What you'll learn:** how to design classes, use inheritance safely (including the MRO), write dunder methods, dataclasses and protocols, and rebuild the interfaces behind scikit-learn estimators and PyTorch modules from scratch.

| | |
|---|---|
| **Difficulty** | 🟢 Beginner → 🟡 Intermediate (a few 🔴 interview-level parts) |
| **Time** | ~4 hours to read and run, +2 hours for exercises and the project |
| **Prerequisites** | [Python Basics](01_Python_Basics.ipynb) (functions, loops) · [Python Built-ins](02_Python_Builtins.ipynb) (lists, dicts, sets) |
| **Tested with** | Python 3.12 · numpy 2.5 · scikit-learn 1.9 · torch 2.14 (optional, one comparison cell) |
| **Interview relevance** | ⭐⭐⭐ High — MRO and `super()`, `__eq__`/`__hash__`, composition vs inheritance, ABC vs Protocol, "write a scikit-learn-compatible estimator" |

## 🤔 What Is Object-Oriented Programming?

Think of a **cookie cutter** and the **cookies** you make with it:

- A **class** is the cookie cutter: a blueprint that says what data an object holds and what it can do.
- An **object** (also called an **instance**) is one cookie: a real thing built from the blueprint, with its own values.
- An **attribute** is a piece of data stored on an object (`model.learning_rate`).
- A **method** is a function that belongs to a class and works on an object (`model.fit(X, y)`).

```
           class TrainingRun            ← blueprint
         ┌──────────────────────┐
         │ name, learning_rate  │  attributes
         │ log_loss(), best()   │  methods
         └──────────┬───────────┘
         ┌──────────┴───────────┐
         ▼                      ▼
  run_a ("baseline", 0.1)   run_b ("small-lr", 0.01)   ← objects (instances)
```

**Object-oriented programming (OOP)** is organising code around such objects: data and the functions that use it live together.

## 🎯 Why It Matters

- **Every ML library you will use is built from classes.** `sklearn` estimators (`fit`/`predict`), PyTorch's `nn.Module`, Hugging Face's `AutoModel.from_pretrained`, pydantic models in FastAPI, LangChain tools — all are OOP designs. Understanding them makes the libraries feel obvious instead of magical.
- **Real ML code bases need structure.** Configs, datasets, trainers, callbacks, model registries — good class design is what keeps a research script from turning into a 3,000-line file.
- **In interviews** you will be asked to explain the MRO and `super()`, why `__eq__` needs `__hash__`, composition vs inheritance, ABC vs Protocol, dataclass vs namedtuple — and to *write* a small class such as a scikit-learn-compatible estimator, an iterable dataset, or a plugin registry. This notebook practises all of it.

## ✅ By the End You Can

- [ ] Write classes with instance/class attributes, properties, and instance/class/static methods
- [ ] Explain inheritance, `super()`, and the C3 method resolution order — and compute an MRO by hand
- [ ] Choose between inheritance, composition, abstract base classes, and protocols
- [ ] Implement dunder methods (`__repr__`, `__eq__` + `__hash__`, `__len__`, `__getitem__`, `__call__`, operators) and dataclasses correctly
- [ ] Build a scikit-learn-style estimator API and a PyTorch-style parameter registry from scratch
- [ ] Spot classic OOP bugs: shared mutable state, broken hashes, missing `super().__init__()`

## 📋 Table of Contents

1. [Classes, Objects, and self](#1.-Classes,-Objects,-and-self-🟢)
2. [Instance vs Class Attributes](#2.-Instance-vs-Class-Attributes-🟢)
3. [Instance, Class, and Static Methods](#3.-Instance,-Class,-and-Static-Methods-🟢)
4. [Properties, Validation, and Encapsulation](#4.-Properties,-Validation,-and-Encapsulation-🟢)
5. [Inheritance and super()](#5.-Inheritance-and-super()-🟡)
6. [Multiple Inheritance and the MRO](#6.-Multiple-Inheritance-and-the-MRO-🟡)
7. [Composition vs Inheritance](#7.-Composition-vs-Inheritance-🟡)
8. [Abstract Base Classes](#8.-Abstract-Base-Classes-🟡)
9. [Protocols and Duck Typing](#9.-Protocols-and-Duck-Typing-🟡)
10. [Dunder (Magic) Methods](#10.-Dunder-(Magic)-Methods-🟡)
11. [Dataclasses](#11.-Dataclasses-🟢)
12. [__slots__ and Memory](#12.-__slots__-and-Memory-🔴)
13. [Design Patterns in ML Code](#13.-Design-Patterns-in-ML-Code-🟡)
- [🔧 Build It From Scratch](#🔧-Build-It-From-Scratch) · [⚠️ Common Pitfalls](#⚠️-Common-Pitfalls) · [🏋️ Practice Exercises](#🏋️-Practice-Exercises) · [🚀 Mini Project](#🚀-Mini-Project:-A-scikit-learn-Style-Pipeline-From-Scratch) · [🎤 Interview Q&A](#🎤-Interview-Q&A) · [🧪 Quick Quiz](#🧪-Quick-Quiz) · [📚 Resources](#📚-Resources) · [📝 Summary](#📝-Summary-Cheat-Sheet)

## ⚙️ Setup

Run the cell below first. Everything except NumPy and scikit-learn is in the standard library; PyTorch is used in **one** optional comparison cell.

The `check()` helper gives you instant feedback on exercises: **✅** correct, **⏳** not attempted yet, **❌** wrong (with a hint).

In [1]:
# %pip install -q "numpy>=2.0" "scikit-learn>=1.5"   # optional: "torch>=2.4"

import copy
import inspect
import json
import sys
import tracemalloc
from abc import ABC, abstractmethod
from collections.abc import Sequence
from dataclasses import FrozenInstanceError, asdict, dataclass, field, replace
from typing import Protocol, runtime_checkable

import numpy as np
import sklearn

print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)


def check(name, got, expected, hint=""):
    """✅ if correct, ⏳ if not attempted yet (None), ❌ AssertionError with a hint otherwise.

    Floats and NumPy arrays are compared with a small tolerance; everything else with ==.
    """
    if got is None or got is ...:
        print(f"⏳ {name}: not attempted yet — replace None with your answer.")
        return
    try:
        if isinstance(expected, (float, np.ndarray)):
            ok = np.shape(got) == np.shape(expected) and np.allclose(got, expected)
        else:
            ok = bool(got == expected)
    except Exception:  # e.g. comparing an array with a list
        ok = False
    assert ok, f"❌ {name}: not quite (got {got!r}). {hint}"
    print(f"✅ {name}: correct!")

Python 3.12.11 | NumPy 2.5.3 | scikit-learn 1.9.1


## 1. Classes, Objects, and self 🟢

- `class TrainingRun:` defines the blueprint.
- `__init__` is the **initializer**: Python calls it automatically right after creating a new object, so you can store starting values.
- `self` is **the specific object a method is working on**. Writing `self.name = name` stores `name` *on that object*.

In [2]:
class TrainingRun:
    """Keeps track of one model-training experiment."""

    def __init__(self, name, learning_rate):
        self.name = name                      # attributes live on THIS object
        self.learning_rate = learning_rate
        self.losses = []                      # a fresh list for every run

    def log_loss(self, loss):
        self.losses.append(loss)

    def best_loss(self):
        return min(self.losses) if self.losses else None


run_a = TrainingRun("baseline", learning_rate=0.1)
run_b = TrainingRun("small-lr", learning_rate=0.01)
for loss in [0.9, 0.6, 0.7]:
    run_a.log_loss(loss)

print(run_a.name, "| best:", run_a.best_loss(), "| losses:", run_a.losses)
print(run_b.name, "| best:", run_b.best_loss(), "| losses:", run_b.losses)
print("type:", type(run_a).__name__, "| isinstance:", isinstance(run_a, TrainingRun))
print("attributes stored on run_a:", vars(run_a))

baseline | best: 0.6 | losses: [0.9, 0.6, 0.7]
small-lr | best: None | losses: []
type: TrainingRun | isinstance: True
attributes stored on run_a: {'name': 'baseline', 'learning_rate': 0.1, 'losses': [0.9, 0.6, 0.7]}


`self` is not magic — it is just the first argument. `run_b.log_loss(0.42)` is shorthand for `TrainingRun.log_loss(run_b, 0.42)`:

In [3]:
TrainingRun.log_loss(run_b, 0.42)             # exactly what run_b.log_loss(0.42) does
print("run_b.losses:", run_b.losses)
print("a bound method remembers its object:", run_b.log_loss.__self__ is run_b)

run_b.losses: [0.42]
a bound method remembers its object: True


### ✍️ Your Turn

Write a class `Counter` whose objects start with `count = 0` and have a method `increment(step=1)` that adds `step` to `count` and returns the new count. Two counters must not share their counts.

In [4]:
Counter = None  # TODO: replace with your class definition
got = None
if Counter is not None:
    c = Counter()
    c.increment()
    c.increment(5)
    got = [c.count, Counter().count]
check("counter", got, [6, 0], hint="Set self.count = 0 in __init__, then do self.count += step in increment().")

⏳ counter: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class Counter:
    def __init__(self):
        self.count = 0

    def increment(self, step=1):
        self.count += step
        return self.count


c = Counter()
c.increment()
c.increment(5)
check("counter", [c.count, Counter().count], [6, 0])
```
</details>

> 💡 **Interview angle:** "What is `self`?" — an explicit reference to the instance; `obj.method(x)` is `type(obj).method(obj, x)`. It is a naming convention, not a keyword.

## 2. Instance vs Class Attributes 🟢

- An **instance attribute** (`self.name = ...`) belongs to one object.
- A **class attribute** (written directly in the class body) is **shared by all objects** of that class.

**Lookup rule:** when you read `obj.attr`, Python looks in the object first (`vars(obj)`), then in its class, then in parent classes. *Assigning* `obj.attr = value` always writes to the object — it never changes the class.

In [5]:
class Model:
    framework = "numpy"                # class attribute: one value shared by all models

    def __init__(self, name):
        self.name = name               # instance attribute: one per object


m1, m2 = Model("a"), Model("b")
print("start                     :", m1.framework, m2.framework, Model.framework)
m1.framework = "torch"                 # creates a NEW instance attribute on m1 that hides the class one
print("after m1.framework='torch':", m1.framework, m2.framework, Model.framework)
print("vars(m1):", vars(m1), "| vars(m2):", vars(m2))
Model.framework = "jax"                # changes the class → everyone who didn't shadow it sees the change
print("after Model.framework='jax':", m1.framework, m2.framework)

start                     : numpy numpy numpy
after m1.framework='torch': torch numpy numpy
vars(m1): {'name': 'a', 'framework': 'torch'} | vars(m2): {'name': 'b'}
after Model.framework='jax': torch jax


**The classic bug:** a *mutable* class attribute (list, dict, set) is one object shared by every instance. Calling `.append` does not assign, so nothing gets shadowed — every instance writes into the same list.

In [6]:
class BuggyExperiment:
    metrics = []                           # ❌ ONE list shared by every experiment

    def __init__(self, name):
        self.name = name

    def log(self, value):
        self.metrics.append(value)         # mutates the shared class list


e1, e2 = BuggyExperiment("e1"), BuggyExperiment("e2")
e1.log(0.91)
print("❌ e2 never logged, but e2.metrics =", e2.metrics, "| same list object:", e1.metrics is e2.metrics)


class Experiment:
    def __init__(self, name):
        self.name = name
        self.metrics = []                  # ✅ a new list for each object

    def log(self, value):
        self.metrics.append(value)


e1, e2 = Experiment("e1"), Experiment("e2")
e1.log(0.91)
print("✅ e2.metrics =", e2.metrics)

❌ e2 never logged, but e2.metrics = [0.91] | same list object: True
✅ e2.metrics = []


> 💡 **Interview angle:** "Why do all my objects share the same list?" — it's a class attribute (or a mutable default argument). Use class attributes only for constants; create mutable state inside `__init__`.

## 3. Instance, Class, and Static Methods 🟢

| Kind | Decorator | First argument | Typical use |
|---|---|---|---|
| Instance method | none | `self` (the object) | normal behaviour that reads/writes object data |
| Class method | `@classmethod` | `cls` (the class) | **alternative constructors** such as `from_json`, `from_pretrained` |
| Static method | `@staticmethod` | nothing | a helper that is related to the class but needs no object or class |

Using `cls(...)` instead of the hard-coded class name means factories automatically work for subclasses.

In [7]:
class ModelConfig:
    VALID_OPTIMIZERS = {"sgd", "adam"}

    def __init__(self, hidden_size, optimizer="adam"):
        if optimizer not in self.VALID_OPTIMIZERS:
            raise ValueError(f"optimizer must be one of {sorted(self.VALID_OPTIMIZERS)}, got {optimizer!r}")
        self.hidden_size = hidden_size
        self.optimizer = optimizer

    def __repr__(self):
        return f"{type(self).__name__}(hidden_size={self.hidden_size}, optimizer={self.optimizer!r})"

    @classmethod
    def from_dict(cls, config):
        return cls(**config)                          # cls is ModelConfig OR a subclass

    @classmethod
    def from_json(cls, text):
        return cls.from_dict(json.loads(text))

    @staticmethod
    def mlp_param_count(n_in, hidden, n_out):
        return (n_in * hidden + hidden) + (hidden * n_out + n_out)   # weights + biases of 2 layers


class BigModelConfig(ModelConfig):
    pass


print(ModelConfig.from_json('{"hidden_size": 64, "optimizer": "sgd"}'))
print(BigModelConfig.from_dict({"hidden_size": 4096}), "← the subclass factory returned a subclass object")
print("params of a 784→64→10 MLP:", ModelConfig.mlp_param_count(784, 64, 10))

ModelConfig(hidden_size=64, optimizer='sgd')
BigModelConfig(hidden_size=4096, optimizer='adam') ← the subclass factory returned a subclass object
params of a 784→64→10 MLP: 50890


### ✍️ Your Turn

Add a class method `from_fahrenheit(cls, f)` to `Temperature` that converts with `(f - 32) * 5 / 9` and returns `cls(celsius)`. The check calls it on the **subclass** `LabTemperature`, so it must use `cls`.

In [8]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    # TODO: add a @classmethod called from_fahrenheit here


class LabTemperature(Temperature):
    pass


got = None
if hasattr(Temperature, "from_fahrenheit"):
    t = LabTemperature.from_fahrenheit(212)
    got = [t.celsius, type(t).__name__]
check("from_fahrenheit", got, [100.0, "LabTemperature"], hint="Decorate with @classmethod and return cls((f - 32) * 5 / 9).")

⏳ from_fahrenheit: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f):
        return cls((f - 32) * 5 / 9)


class LabTemperature(Temperature):
    pass


t = LabTemperature.from_fahrenheit(212)
check("from_fahrenheit", [t.celsius, type(t).__name__], [100.0, "LabTemperature"])
```

Writing `return Temperature(...)` would also convert correctly, but `LabTemperature.from_fahrenheit` would then return the wrong type.
</details>

> 💡 **Interview angle:** "`@classmethod` vs `@staticmethod`?" — a class method receives the class, so it can build instances (and subclasses); a static method receives nothing and is just a namespaced function. Name a real example: `dict.fromkeys`, `datetime.fromisoformat`, `AutoModel.from_pretrained`.

## 4. Properties, Validation, and Encapsulation 🟢

**Encapsulation** means hiding internal details behind a small public interface. Python does it by *convention*:

| Name | Meaning |
|---|---|
| `learning_rate` | public |
| `_learning_rate` | "internal — please don't touch" (nothing stops you) |
| `__token` | **name mangling**: Python renames it to `_ClassName__token` to avoid clashes in subclasses — not real privacy |

A **property** looks like a plain attribute to the caller but runs code when read or written — perfect for validation and computed values. You can start with a plain attribute and switch to a property later **without changing any calling code**.

In [9]:
class Optimizer:
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate        # goes through the setter below → validated here too
        self._steps = 0

    @property
    def learning_rate(self):
        return self._learning_rate                # the real value lives in a differently named attribute

    @learning_rate.setter
    def learning_rate(self, value):
        if not value > 0:
            raise ValueError(f"learning_rate must be > 0, got {value}")
        self._learning_rate = float(value)

    @property
    def steps(self):                              # read-only: no setter defined
        return self._steps

    def step(self):
        self._steps += 1


opt = Optimizer(0.01)
opt.step()
opt.step()
opt.learning_rate = 0.001
print("lr:", opt.learning_rate, "| steps:", opt.steps)

for attempt in (lambda: setattr(opt, "learning_rate", -1), lambda: Optimizer(0), lambda: setattr(opt, "steps", 100)):
    try:
        attempt()
    except (ValueError, AttributeError) as err:
        print(f"{type(err).__name__}: {err}")

lr: 0.001 | steps: 2
ValueError: learning_rate must be > 0, got -1
ValueError: learning_rate must be > 0, got 0
AttributeError: property 'steps' of 'Optimizer' object has no setter


In [10]:
class Secretive:
    def __init__(self):
        self.__token = "abc123"


s = Secretive()
print("stored under the mangled name:", list(vars(s)))
print("still reachable (so not truly private):", s._Secretive__token)

stored under the mangled name: ['_Secretive__token']
still reachable (so not truly private): abc123


> 💡 **Interview angle:** "Does Python have private attributes?" — no; `_x` is a convention and `__x` only name-mangles. Properties let you add validation later while keeping the public API unchanged.

## 5. Inheritance and super() 🟡

**Inheritance** lets a *child* class (subclass) reuse and extend a *parent* class (base class). Use it for true **"is-a"** relationships: a `MeanRegressor` *is a* model.

- **Overriding** = redefining a parent method in the child.
- **`super()`** gives you the parent's version, so you *extend* instead of copy-pasting.
- **Liskov substitution principle:** a subclass should work anywhere its parent is expected.

In [11]:
class BaseModel:
    def __init__(self, name):
        self.name = name
        self.is_fitted = False

    def fit(self, X, y):
        self.is_fitted = True
        return self                                     # returning self enables model.fit(X, y).predict(X)

    def describe(self):
        return f"{type(self).__name__}({self.name!r}, fitted={self.is_fitted})"


class MeanRegressor(BaseModel):
    """Predicts the training mean — the simplest possible regression baseline."""

    def __init__(self, name="mean-baseline"):
        super().__init__(name)                          # let the parent set up its attributes first

    def fit(self, X, y):
        self.mean_ = float(np.mean(y))
        return super().fit(X, y)                        # reuse the parent's bookkeeping

    def predict(self, X):
        return np.full(len(X), self.mean_)

    def describe(self):
        return super().describe() + f", mean={getattr(self, 'mean_', None)}"


X_toy = np.arange(8).reshape(4, 2)
y_toy = np.array([1.0, 2.0, 3.0, 6.0])

model = MeanRegressor()
print(model.describe())
print("predictions:", model.fit(X_toy, y_toy).predict(X_toy[:2]), "|", model.describe())
print("isinstance of parent:", isinstance(model, BaseModel), "| bases:", MeanRegressor.__bases__)

MeanRegressor('mean-baseline', fitted=False), mean=None
predictions: [3. 3.] | MeanRegressor('mean-baseline', fitted=True), mean=3.0
isinstance of parent: True | bases: (<class '__main__.BaseModel'>,)


> 💡 **Interview angle:** "When should you use inheritance?" — for a genuine is-a relationship where the subclass can replace the parent everywhere (Liskov). If you only want to *reuse some code*, composition (section 7) is usually the better tool.

## 6. Multiple Inheritance and the MRO 🟡

A class can have several parents. Then Python needs a rule for *where to look first*. That order is the **method resolution order (MRO)**, computed with the **C3 linearization** algorithm (used since Python 2.3). C3 guarantees:

1. a child comes before its parents,
2. parents keep the order you listed them in `class Child(A, B)`,
3. every parent's own MRO order is preserved.

```
           Base              MixinTrainer(LoggingMixin, CheckpointMixin)
          /    \
 LoggingMixin  CheckpointMixin      MRO: MixinTrainer → LoggingMixin → CheckpointMixin → Base → object
          \    /
        MixinTrainer        ← the "diamond"
```

**The key insight:** `super()` does **not** mean "my parent". It means **"the next class in the MRO of the object I'm working on"**.

In [12]:
class Base:
    def __init__(self):
        print("   Base.__init__ (runs once)")


class LoggingMixin(Base):
    def __init__(self):
        print("   LoggingMixin.__init__ → super()")
        super().__init__()


class CheckpointMixin(Base):
    def __init__(self):
        print("   CheckpointMixin.__init__ → super()")
        super().__init__()


class MixinTrainer(LoggingMixin, CheckpointMixin):
    def __init__(self):
        print("   MixinTrainer.__init__ → super()")
        super().__init__()


print("MRO:", " → ".join(k.__name__ for k in MixinTrainer.__mro__))
MixinTrainer()
print("LoggingMixin's parent is Base, yet its super() called CheckpointMixin — the next class in THIS object's MRO.")

MRO: MixinTrainer → LoggingMixin → CheckpointMixin → Base → object
   MixinTrainer.__init__ → super()
   LoggingMixin.__init__ → super()
   CheckpointMixin.__init__ → super()
   Base.__init__ (runs once)
LoggingMixin's parent is Base, yet its super() called CheckpointMixin — the next class in THIS object's MRO.


When no order can satisfy all three rules, Python refuses to create the class:

In [13]:
class X:
    pass


class Y(X):
    pass


try:
    class Z(X, Y):          # asks for X before Y, but Y must come before its parent X
        pass
except TypeError as err:
    print("TypeError:", err)

TypeError: Cannot create a consistent method resolution
order (MRO) for bases X, Y


**Where you'll see this for real:** scikit-learn puts *mixins* (small classes that add one capability) on the left and `BaseEstimator` on the right, e.g. `class MyClassifier(ClassifierMixin, BaseEstimator)`, so the mixin's methods win.

In [14]:
from sklearn.linear_model import LogisticRegression

print(" → ".join(k.__name__ for k in LogisticRegression.__mro__))

LogisticRegression → CallbackSupportMixin → LinearClassifierMixin → ClassifierMixin → SparseCoefMixin → BaseEstimator → ReprHTMLMixin → _HTMLDocumentationLinkMixin → _MetadataRequester → object


> 💡 **Interview angle:** "In `class D(B, C)` with `B` and `C` both inheriting `A`, what does `super()` inside `B` call?" — `C`, because `D.__mro__` is `D, B, C, A, object`. Say the words *C3 linearization* and *cooperative* `super()` (every class calls `super()` so each runs exactly once).

## 7. Composition vs Inheritance 🟡

- **Inheritance** = "is-a". Reuses code by *becoming* the parent. Tightly couples child to parent.
- **Composition** = "has-a". An object *holds* other objects and delegates work to them. You can swap parts at runtime.

"Favour composition over inheritance" (from the *Design Patterns* book) matters when behaviours vary independently. With 3 model types × 3 loggers × 2 early-stopping rules, inheritance needs a subclass per combination; composition needs one small class per behaviour:

In [15]:
n_models, n_loggers, n_stoppers = 3, 3, 2
print(f"inheritance: {n_models * n_loggers * n_stoppers} combined subclasses | composition: {n_models + n_loggers + n_stoppers} small classes")


class ListLogger:
    def __init__(self):
        self.lines = []

    def log(self, message):
        self.lines.append(message)


class EarlyStopping:
    def __init__(self, patience):
        self.patience = patience
        self.best = float("inf")
        self.bad_epochs = 0

    def should_stop(self, loss):
        if loss < self.best:
            self.best, self.bad_epochs = loss, 0
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience


class SimpleTrainer:
    """HAS a logger and HAS a stopping rule — both are passed in (dependency injection)."""

    def __init__(self, logger, stopper):
        self.logger = logger
        self.stopper = stopper

    def run(self, epoch_losses):
        for epoch, loss in enumerate(epoch_losses):
            self.logger.log(f"epoch {epoch}: loss={loss}")
            if self.stopper.should_stop(loss):
                self.logger.log(f"early stop at epoch {epoch}")
                break
        return epoch


losses = [1.0, 0.8, 0.75, 0.76, 0.77, 0.74]
logger = ListLogger()
print("patience=2 stopped at epoch:", SimpleTrainer(logger, EarlyStopping(patience=2)).run(losses))
print("last log line:", logger.lines[-1])
print("patience=3 stopped at epoch:", SimpleTrainer(ListLogger(), EarlyStopping(patience=3)).run(losses))

inheritance: 18 combined subclasses | composition: 8 small classes
patience=2 stopped at epoch: 4
last log line: early stop at epoch 4
patience=3 stopped at epoch: 5


### ✍️ Your Turn

Without touching `SimpleTrainer`, write a class `NeverStop` whose `should_stop(loss)` always returns `False`, and run the trainer with it on `losses`. The returned last epoch should be `5`.

In [16]:
NeverStop = None  # TODO: a class with should_stop(self, loss) that always returns False
last_epoch = None
if NeverStop is not None:
    last_epoch = SimpleTrainer(ListLogger(), NeverStop()).run(losses)
check("never_stop", last_epoch, 5, hint="The trainer only needs an object with a should_stop method — no inheritance required.")

⏳ never_stop: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class NeverStop:
    def should_stop(self, loss):
        return False


last_epoch = SimpleTrainer(ListLogger(), NeverStop()).run(losses)
check("never_stop", last_epoch, 5)
```

That's the **open/closed principle** in action: the trainer is *closed* for modification but *open* for extension.
</details>

> 💡 **Interview angle:** "Composition or inheritance for a trainer's logging?" — composition: inject a logger object, so you can swap in W&B / MLflow / a test fake without a new subclass. Keep inheritance for framework extension points like `nn.Module` or `BaseEstimator`.

## 8. Abstract Base Classes 🟡

An **abstract base class (ABC)** is a class you *can't instantiate*; it declares methods (with `@abstractmethod`) that every concrete subclass **must** implement. Python enforces this when you try to create an object. It's a *contract* plus shared code.

In [17]:
class Estimator(ABC):
    @abstractmethod
    def fit(self, X, y): ...

    @abstractmethod
    def predict(self, X): ...

    def fit_predict(self, X, y):                   # concrete method built on the abstract ones
        return self.fit(X, y).predict(X)


class HalfDone(Estimator):
    def fit(self, X, y):
        return self


for cls in (Estimator, HalfDone):
    try:
        cls()
    except TypeError as err:
        print("TypeError:", err)


class MajorityClassifier(Estimator):
    def fit(self, X, y):
        values, counts = np.unique(y, return_counts=True)
        self.majority_ = values[counts.argmax()]
        return self

    def predict(self, X):
        return np.full(len(X), self.majority_)


y_toy_cls = np.array([0, 1, 1, 1])
print("fit_predict:", MajorityClassifier().fit_predict(X_toy, y_toy_cls), "| is an Estimator:", isinstance(MajorityClassifier(), Estimator))

TypeError: Can't instantiate abstract class Estimator without an implementation for abstract methods 'fit', 'predict'
TypeError: Can't instantiate abstract class HalfDone without an implementation for abstract method 'predict'
fit_predict: [1 1 1 1] | is an Estimator: True


The standard library's `collections.abc` ABCs also hand you **free methods**: implement `__getitem__` and `__len__` on a `Sequence` and you get `in`, iteration, `reversed`, `.index` and `.count`.

In [18]:
class Batch(Sequence):
    def __init__(self, items):
        self._items = list(items)

    def __getitem__(self, i):
        return self._items[i]

    def __len__(self):
        return len(self._items)


batch = Batch(["cat", "dog", "bird"])
print("'dog' in batch:", "dog" in batch, "| reversed:", list(reversed(batch)), "| index('bird'):", batch.index("bird"))

'dog' in batch: True | reversed: ['bird', 'dog', 'cat'] | index('bird'): 2


> 💡 **Interview angle:** "What does `@abstractmethod` actually enforce?" — only that a subclass *defines* a method with that name before it can be instantiated; it doesn't check the signature or behaviour.

## 9. Protocols and Duck Typing 🟡

**Duck typing:** "if it walks like a duck and quacks like a duck, it's a duck." Python code usually doesn't care about an object's *class*, only that it has the methods you call.

- An ABC is **nominal**: you must *inherit* from it.
- A `typing.Protocol` is **structural**: *any* class with matching methods fits, no inheritance needed. Protocols are checked by static type checkers (mypy, pyright) before your code runs.

In [19]:
class SupportsPredict(Protocol):
    def predict(self, X): ...


def accuracy(model: SupportsPredict, X, y) -> float:
    """Works with ANY object that has .predict — ours, scikit-learn's, or a wrapper around a remote API."""
    return float(np.mean(model.predict(X) == y))


class ConstantModel:                       # inherits from nothing
    def predict(self, X):
        return np.ones(len(X), dtype=int)


from sklearn.dummy import DummyClassifier

print("our ABC subclass :", accuracy(MajorityClassifier().fit(X_toy, y_toy_cls), X_toy, y_toy_cls))
print("scikit-learn     :", accuracy(DummyClassifier(strategy="most_frequent").fit(X_toy, y_toy_cls), X_toy, y_toy_cls))
print("plain duck-typed :", accuracy(ConstantModel(), X_toy, y_toy_cls))

our ABC subclass : 0.75
scikit-learn     : 0.75
plain duck-typed : 0.75


`@runtime_checkable` lets you use `isinstance` with a protocol — but it **only checks that the names exist**, not their signatures or even that they are callable:

In [20]:
@runtime_checkable
class SupportsFit(Protocol):
    def fit(self, X, y): ...


class LooksRight:
    fit = "not even a function"


print("DummyClassifier:", isinstance(DummyClassifier(), SupportsFit))
print("ConstantModel  :", isinstance(ConstantModel(), SupportsFit))
print("LooksRight     :", isinstance(LooksRight(), SupportsFit), "← passes, even though fit is a string!")

DummyClassifier: True
ConstantModel  : False
LooksRight     : True ← passes, even though fit is a string!


### ✍️ Your Turn

Write a class `ThresholdModel` (no inheritance) that takes `threshold` in `__init__` and whose `predict(X)` returns `1` where the **first column** of `X` is greater than `threshold`, else `0`. With `threshold=1` it should get accuracy `1.0` on `X_toy, y_toy_cls`.

In [21]:
ThresholdModel = None  # TODO: replace with your class
score = None
if ThresholdModel is not None:
    score = accuracy(ThresholdModel(threshold=1), X_toy, y_toy_cls)
check("threshold_model", score, 1.0, hint="predict can return (X[:, 0] > self.threshold).astype(int).")

⏳ threshold_model: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class ThresholdModel:
    def __init__(self, threshold):
        self.threshold = threshold

    def predict(self, X):
        return (X[:, 0] > self.threshold).astype(int)


score = accuracy(ThresholdModel(threshold=1), X_toy, y_toy_cls)
check("threshold_model", score, 1.0)
```
</details>

> 💡 **Interview angle:** "ABC or Protocol?" — ABC when you own the hierarchy and want runtime enforcement plus shared methods; Protocol when you want to accept third-party objects (like any scikit-learn model) without forcing them to inherit from your class.

## 10. Dunder (Magic) Methods 🟡

**Dunder** = "double underscore". These methods let your objects plug into Python's syntax and built-ins:

| You write | Python calls |
|---|---|
| `repr(v)`, the REPL, inside lists | `v.__repr__()` |
| `v == w` | `v.__eq__(w)` |
| `{v}`, `dict[v]` | `v.__hash__()` |
| `len(v)`, `v[i]`, `for x in v` | `__len__`, `__getitem__`, `__iter__` |
| `v + w`, `2 * v`, `v @ w` | `__add__`, `__rmul__`, `__matmul__` |
| `if v:` | `__bool__` (falls back to `__len__`) |
| `model(x)` | `model.__call__(x)` |

Return **`NotImplemented`** from an operator when you don't know how to handle the other type: Python then tries the other object's reflected method and finally raises a clear `TypeError`.

In [22]:
class Vector:
    def __init__(self, *components):
        self.components = tuple(float(c) for c in components)

    def __repr__(self):
        return f"Vector{self.components}"

    def __len__(self):
        return len(self.components)

    def __getitem__(self, i):
        return self.components[i]

    def __iter__(self):
        return iter(self.components)

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(*(a + b for a, b in zip(self, other, strict=True)))

    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector(*(a * scalar for a in self))

    __rmul__ = __mul__                          # so that 2 * v works as well as v * 2

    def __matmul__(self, other):                # dot product with @
        return sum(a * b for a, b in zip(self, other, strict=True))

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return self.components == other.components

    def __hash__(self):                         # equal vectors → equal hashes
        return hash(self.components)

    def __bool__(self):
        return any(self.components)


v, w = Vector(1, 2), Vector(3, 4)
print("v + w  :", v + w, "| 2 * v:", 2 * v, "| v @ w:", v @ w)
print("len, v[1], list:", len(v), v[1], list(v))
print("equal & hashable:", v == Vector(1, 2), "| set size:", len({v, Vector(1, 2), w}))
print("bool(zero vector):", bool(Vector(0, 0)))
try:
    v + 1
except TypeError as err:
    print("TypeError:", err)

v + w  : Vector(4.0, 6.0) | 2 * v: Vector(2.0, 4.0) | v @ w: 11.0
len, v[1], list: 2 2.0 [1.0, 2.0]
equal & hashable: True | set size: 2
bool(zero vector): False
TypeError: unsupported operand type(s) for +: 'Vector' and 'int'


**The `__eq__` + `__hash__` rule:** objects that compare equal **must** have the same hash. So if you define `__eq__` and not `__hash__`, Python sets `__hash__ = None` and your objects can't go in sets or be dict keys:

In [23]:
class Token:
    def __init__(self, text):
        self.text = text

    def __eq__(self, other):
        return isinstance(other, Token) and self.text == other.text


print("Token.__hash__ is", Token.__hash__)
try:
    {Token("hi")}
except TypeError as err:
    print("TypeError:", err)

Token.__hash__ is None
TypeError: unhashable type: 'Token'


`__call__` makes an object behave like a function. PyTorch uses exactly this: `model(x)` runs `nn.Module.__call__`, which runs hooks and then your `forward`.

In [24]:
class Standardize:
    def __init__(self, mean, std):
        self.mean, self.std = mean, std

    def __call__(self, x):
        return (x - self.mean) / self.std


normalize = Standardize(mean=10.0, std=2.0)
print("callable object:", normalize(np.array([8.0, 10.0, 14.0])), "| callable():", callable(normalize))

callable object: [-1.  0.  2.] | callable(): True


### ✍️ Your Turn

PyTorch datasets are just classes with `__len__` and `__getitem__`. Write `WindowDataset(series, window)` that frames a time series for forecasting:
- `len(ds)` is `len(series) - window`
- `ds[i]` returns a tuple `(series[i:i + window], series[i + window])` — the inputs and the value to predict.

In [25]:
WindowDataset = None  # TODO: replace with your class
series = [10, 20, 30, 40, 50]
got = None
if WindowDataset is not None:
    ds = WindowDataset(series, window=2)
    got = [len(ds), ds[0], ds[len(ds) - 1]]
check("window_dataset", got, [3, ([10, 20], 30), ([30, 40], 50)],
      hint="__len__ returns len(self.series) - self.window; __getitem__ slices self.series.")

⏳ window_dataset: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class WindowDataset:
    def __init__(self, series, window):
        self.series = series
        self.window = window

    def __len__(self):
        return len(self.series) - self.window

    def __getitem__(self, i):
        if not 0 <= i < len(self):
            raise IndexError(i)          # lets `for x in ds` stop at the end
        return self.series[i:i + self.window], self.series[i + self.window]


series = [10, 20, 30, 40, 50]
ds = WindowDataset(series, window=2)
check("window_dataset", [len(ds), ds[0], ds[len(ds) - 1]], [3, ([10, 20], 30), ([30, 40], 50)])
```

Raising `IndexError` matters: `for x in ds` keeps calling `__getitem__(0), (1), …` until it sees `IndexError`.
</details>

> 💡 **Interview angle:** "Why return `NotImplemented` instead of raising?" — it lets Python try the other operand's reflected method (`__radd__`) before raising a proper `TypeError`, which keeps operators composable with other types.

## 11. Dataclasses 🟢

`@dataclass` writes the boring parts (`__init__`, `__repr__`, `__eq__`) from **type-annotated fields**. Perfect for configs and records.

- Mutable defaults need `field(default_factory=list)` — Python raises an error otherwise.
- `__post_init__` runs after the generated `__init__`: a good place for validation.
- Annotations are **not** enforced at runtime (use pydantic when data comes from outside).

In [26]:
@dataclass
class TrainConfig:
    model_name: str
    learning_rate: float = 3e-4
    epochs: int = 10
    tags: list[str] = field(default_factory=list)       # a new list per config

    def __post_init__(self):
        if self.learning_rate <= 0:
            raise ValueError(f"learning_rate must be > 0, got {self.learning_rate}")


cfg = TrainConfig("resnet18", epochs=5)
print(cfg)
print("value equality:", cfg == TrainConfig("resnet18", epochs=5), "| as dict:", asdict(cfg))

try:
    TrainConfig("broken", learning_rate=0)
except ValueError as err:
    print("ValueError:", err)

try:
    @dataclass
    class BadConfig:
        tags: list = []
except ValueError as err:
    print("ValueError:", err)

TrainConfig(model_name='resnet18', learning_rate=0.0003, epochs=5, tags=[])
value equality: True | as dict: {'model_name': 'resnet18', 'learning_rate': 0.0003, 'epochs': 5, 'tags': []}
ValueError: learning_rate must be > 0, got 0
ValueError: mutable default <class 'list'> for field tags is not allowed: use default_factory


`frozen=True` makes instances immutable **and hashable**; `order=True` adds `<`, `>` by comparing fields in order — handy for version numbers, where string comparison is wrong:

In [27]:
@dataclass(frozen=True, order=True)
class ModelVersion:
    major: int
    minor: int
    patch: int = 0


versions = [ModelVersion(1, 10), ModelVersion(1, 2, 5), ModelVersion(0, 9)]
print("sorted:", sorted(versions))
print("string compare says '1.10' < '1.2':", "1.10" < "1.2", "| dataclass says 1.10 > 1.2:", ModelVersion(1, 10) > ModelVersion(1, 2))

current = ModelVersion(1, 2)
try:
    current.minor = 3
except FrozenInstanceError as err:
    print("FrozenInstanceError:", err)
print("copy with a change:", replace(current, minor=3), "| usable as a dict key:", {current: "production"})

sorted: [ModelVersion(major=0, minor=9, patch=0), ModelVersion(major=1, minor=2, patch=5), ModelVersion(major=1, minor=10, patch=0)]
string compare says '1.10' < '1.2': True | dataclass says 1.10 > 1.2: True
FrozenInstanceError: cannot assign to field 'minor'
copy with a change: ModelVersion(major=1, minor=3, patch=0) | usable as a dict key: {ModelVersion(major=1, minor=2, patch=0): 'production'}


| Need | Pick |
|---|---|
| Small immutable record, unpacking/indexing like a tuple | `typing.NamedTuple` |
| Config / internal record with defaults, optional immutability | `@dataclass` (`frozen=True`, `slots=True`, `kw_only=True`) |
| Parsing and validating untrusted input (JSON, API requests) | pydantic `BaseModel` |
| Rich behaviour and invariants | a regular class |

### ✍️ Your Turn

Create a **frozen** dataclass `Hyperparams` with fields `lr: float` and `batch_size: int = 32`. Then make `base = Hyperparams(0.01)`, `bigger` = a copy of `base` with `batch_size=64` (use `replace`), and `n_unique` = the number of distinct objects in `[Hyperparams(0.01), Hyperparams(0.01), Hyperparams(0.1)]` (use a set).

In [28]:
Hyperparams = None  # TODO: frozen dataclass
base = bigger = n_unique = None  # TODO
got = None if n_unique is None else [bigger.batch_size, base.batch_size, n_unique]
check("hyperparams", got, [64, 32, 2], hint="@dataclass(frozen=True) makes it hashable, so set() can deduplicate.")

⏳ hyperparams: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
@dataclass(frozen=True)
class Hyperparams:
    lr: float
    batch_size: int = 32


base = Hyperparams(0.01)
bigger = replace(base, batch_size=64)
n_unique = len({Hyperparams(0.01), Hyperparams(0.01), Hyperparams(0.1)})
check("hyperparams", [bigger.batch_size, base.batch_size, n_unique], [64, 32, 2])
```

Hashable configs are useful as cache keys, e.g. "have I already trained this hyperparameter combination?"
</details>

> 💡 **Interview angle:** "Why is a plain `@dataclass` unhashable?" — `eq=True` generates `__eq__` over mutable fields, so Python sets `__hash__` to `None`; use `frozen=True` (or `unsafe_hash=True` if you really know the fields won't change).

## 12. __slots__ and Memory 🔴

Normally every object stores its attributes in a per-object dictionary (`__dict__`), so you can add attributes at any time. Declaring **`__slots__`** fixes the attribute names in advance: no `__dict__`, less memory per object, but no new attributes.

`sys.getsizeof` is misleading here (it doesn't count the separate dict), so we measure real allocations with `tracemalloc`:

In [29]:
class PointDict:
    def __init__(self, x, y):
        self.x, self.y = x, y


class PointSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x, self.y = x, y


def bytes_per_object(cls, n=200_000):
    tracemalloc.start()
    before, _ = tracemalloc.get_traced_memory()
    objects = [cls(1.0, 2.0) for _ in range(n)]          # same float objects → we measure only the instances
    after, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    del objects
    return (after - before) / n


dict_bytes, slot_bytes = bytes_per_object(PointDict), bytes_per_object(PointSlots)
print(f"regular object: {dict_bytes:5.1f} bytes | __slots__ object: {slot_bytes:5.1f} bytes (includes the list's 8-byte pointer)")
print(f"→ __slots__ saves {1 - slot_bytes / dict_bytes:.0%} per object, about {(dict_bytes - slot_bytes) * 10_000_000 / 1e6:.0f} MB per 10 million objects")
print("sys.getsizeof says:", sys.getsizeof(PointDict(1.0, 2.0)), "vs", sys.getsizeof(PointSlots(1.0, 2.0)), "← doesn't show the difference")

p = PointSlots(1.0, 2.0)
print("has __dict__:", hasattr(p, "__dict__"))
try:
    p.z = 3.0
except AttributeError as err:
    print("AttributeError:", err)

regular object:  88.1 bytes | __slots__ object:  56.1 bytes (includes the list's 8-byte pointer)
→ __slots__ saves 36% per object, about 320 MB per 10 million objects
sys.getsizeof says: 48 vs 48 ← doesn't show the difference
has __dict__: False
AttributeError: 'PointSlots' object has no attribute 'z'


**When is it worth it?** Only when you create *millions* of small objects (tokens, graph nodes, events). Python 3.11+ already made regular instance dicts much more compact, so savings are smaller than older blog posts claim — always measure. The dataclass shortcut is `@dataclass(slots=True)`.

> 💡 **Interview angle:** "What does `__slots__` do and what's the catch?" — removes the per-instance `__dict__` (less memory, slightly faster attribute access); you can't add new attributes, every class in the hierarchy needs `__slots__` to get the full benefit, and you must add `"__weakref__"` if you need weak references.

## 13. Design Patterns in ML Code 🟡

A **design pattern** is a named, reusable solution to a common design problem. Three show up constantly in ML code bases:

| Pattern | Idea | Where you've seen it |
|---|---|---|
| **Strategy** | pass the varying behaviour in as an object or function | `scoring=` in scikit-learn, loss functions, optimizers |
| **Factory / registry** | build objects from a *name* in a config | `AutoModel.from_pretrained`, `timm.create_model`, Hydra configs |
| **Template method** | a base class fixes the algorithm skeleton; subclasses fill in steps via hooks | Keras callbacks, PyTorch Lightning hooks |

**Honest caveats:** in Python a strategy is often *just a function*; don't create a class that has one method and no state. Avoid the Singleton pattern (a module-level object is simpler and global state makes tests painful). Patterns are vocabulary for talking about design, not goals.

In [30]:
# Strategy — the metric is a plain function passed in
def mse(y, p):
    return float(np.mean((y - p) ** 2))


def mae(y, p):
    return float(np.mean(np.abs(y - p)))


def evaluate(model, X, y, metric):
    return metric(y, model.predict(X))


baseline = MeanRegressor().fit(X_toy, y_toy)
for metric in (mse, mae):
    print(f"{metric.__name__}: {evaluate(baseline, X_toy, y_toy, metric):.3f}")

mse: 3.500
mae: 1.500


In [31]:
# Factory / registry — map config names to classes without an if/elif chain
MODEL_REGISTRY = {}


def register(name):
    def decorator(cls):
        if name in MODEL_REGISTRY:
            raise KeyError(f"model {name!r} is already registered")
        MODEL_REGISTRY[name] = cls
        return cls                      # return the class unchanged so it can still be used normally
    return decorator


@register("mean")
class MeanBaseline(MeanRegressor):
    pass


@register("majority")
class MajorityBaseline(MajorityClassifier):
    pass


def build_model(model_type, **kwargs):        # not called `name`: models may take a `name` kwarg themselves
    if model_type not in MODEL_REGISTRY:
        raise ValueError(f"unknown model {model_type!r}; choose from {sorted(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[model_type](**kwargs)


config = json.loads('{"model": "mean", "params": {"name": "from-config"}}')
built = build_model(config["model"], **config["params"])
print("built from config:", built.describe())
try:
    build_model("xgboost")
except ValueError as err:
    print("ValueError:", err)

built from config: MeanBaseline('from-config', fitted=False), mean=None
ValueError: unknown model 'xgboost'; choose from ['majority', 'mean']


The same registry can fill itself automatically with `__init_subclass__`, a hook Python calls whenever a subclass is created:

In [32]:
class AutoRegistered:
    registry = {}                                   # a class-level dict shared on purpose: it IS the registry

    def __init_subclass__(cls, key=None, **kwargs):
        super().__init_subclass__(**kwargs)
        AutoRegistered.registry[key or cls.__name__.lower()] = cls


class SGDOptimizer(AutoRegistered, key="sgd"):
    pass


class AdamOptimizer(AutoRegistered, key="adam"):
    pass


print({name: cls.__name__ for name, cls in AutoRegistered.registry.items()})

{'sgd': 'SGDOptimizer', 'adam': 'AdamOptimizer'}


In [33]:
# Template method — the base class owns the loop; subclasses fill in one step and optional hooks
class BaseTrainer(ABC):
    def fit(self, data, epochs):
        self.history = []
        self.on_train_start()
        for epoch in range(epochs):
            loss = self.train_one_epoch(data)       # the step each subclass must provide
            self.history.append(loss)
            self.on_epoch_end(epoch, loss)          # optional hook
        return self

    def on_train_start(self):
        pass

    def on_epoch_end(self, epoch, loss):
        pass

    @abstractmethod
    def train_one_epoch(self, data): ...


class SlopeTrainer(BaseTrainer):
    """Fits y ≈ w·x with gradient descent."""

    def __init__(self, lr):
        self.lr = lr

    def on_train_start(self):
        self.w = 0.0

    def train_one_epoch(self, data):
        x, y = data
        error = self.w * x - y
        self.w -= self.lr * 2 * np.mean(error * x)
        return float(np.mean(error ** 2))

    def on_epoch_end(self, epoch, loss):
        if epoch % 10 == 0:
            print(f"   epoch {epoch:2d}: loss={loss:.4f}  w={self.w:.3f}")


# a controlled synthetic example (we want to know the true answer): y = 3x + small noise
x_line = rng.uniform(-1, 1, size=100)
y_line = 3 * x_line + rng.normal(scale=0.1, size=100)
trainer = SlopeTrainer(lr=0.5).fit((x_line, y_line), epochs=30)
print(f"learned w = {trainer.w:.3f} (true 3) | loss fell from {trainer.history[0]:.3f} to {trainer.history[-1]:.4f}")

   epoch  0: loss=2.6913  w=0.893
   epoch 10: loss=0.0119  w=2.942
   epoch 20: loss=0.0096  w=3.002
learned w = 3.004 (true 3) | loss fell from 2.691 to 0.0096


> 💡 **Interview angle:** "How would you let teammates add a new model without editing the training script?" — a registry (decorator or `__init_subclass__`) keyed by a config name, plus a small protocol (`fit`/`predict`) the trainer relies on. Mention the gotcha: a class only registers once its module has been imported.

## 🔧 Build It From Scratch

Two interfaces you use every day, rebuilt with plain Python — then checked against the real libraries.

### 1) A scikit-learn-style estimator API

scikit-learn's rules for estimators (from its developer guide):
1. `__init__` **only stores** hyperparameters, each under an attribute with the **same name** — no validation, no work.
2. `fit(X, y)` learns, stores results in attributes **ending with `_`** (like `coef_`), and **returns `self`**.
3. `get_params()` reads the hyperparameter names from the `__init__` signature; `set_params(**p)` changes them.
4. `clone(est)` = a fresh, unfitted estimator with the same hyperparameters, built via `get_params()`.

Rule 1 exists because of rule 4: `clone` and `GridSearchCV` rebuild estimators by calling `__init__` with `get_params()`, then `set_params` — any logic hidden in `__init__` would be skipped or break that.

In [34]:
class MiniBaseEstimator:
    @classmethod
    def _param_names(cls):
        signature = inspect.signature(cls.__init__)
        return sorted(p.name for p in signature.parameters.values()
                      if p.name != "self" and p.kind not in (p.VAR_POSITIONAL, p.VAR_KEYWORD))

    def get_params(self, deep=True):
        return {name: getattr(self, name) for name in self._param_names()}

    def set_params(self, **params):
        valid = self._param_names()
        for key, value in params.items():
            if key not in valid:
                raise ValueError(f"invalid parameter {key!r} for {type(self).__name__}; valid: {valid}")
            setattr(self, key, value)
        return self

    def __repr__(self):
        args = ", ".join(f"{k}={v!r}" for k, v in self.get_params(deep=False).items())
        return f"{type(self).__name__}({args})"

    def _check_is_fitted(self):
        fitted = any(k.endswith("_") and not k.startswith("__") for k in vars(self))
        if not fitted:
            raise RuntimeError(f"This {type(self).__name__} instance is not fitted yet. Call 'fit' first.")


def mini_clone(obj):
    """A new unfitted copy: estimators are rebuilt from their params; lists/tuples are cloned element-wise."""
    if isinstance(obj, (list, tuple)):
        return type(obj)(mini_clone(item) for item in obj)
    if hasattr(obj, "get_params") and not isinstance(obj, type):
        return type(obj)(**{k: mini_clone(v) for k, v in obj.get_params(deep=False).items()})
    return copy.deepcopy(obj)


class RidgeRegression(MiniBaseEstimator):
    """Linear regression with an L2 penalty, solved in closed form."""

    def __init__(self, alpha=1.0, fit_intercept=True):
        self.alpha = alpha
        self.fit_intercept = fit_intercept

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        x_mean = X.mean(axis=0) if self.fit_intercept else np.zeros(X.shape[1])
        y_mean = y.mean() if self.fit_intercept else 0.0
        Xc = X - x_mean                                     # centring means the intercept isn't penalised
        A = Xc.T @ Xc + self.alpha * np.eye(X.shape[1])
        self.coef_ = np.linalg.solve(A, Xc.T @ (y - y_mean))
        self.intercept_ = float(y_mean - x_mean @ self.coef_)
        return self

    def predict(self, X):
        self._check_is_fitted()
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

In [35]:
from sklearn.base import clone
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge

X_dia, y_dia = load_diabetes(return_X_y=True)

ours = RidgeRegression(alpha=0.5)
try:
    ours.predict(X_dia)
except RuntimeError as err:
    print("before fit →", err)

ours.fit(X_dia, y_dia)
theirs = Ridge(alpha=0.5).fit(X_dia, y_dia)
assert np.allclose(ours.coef_, theirs.coef_) and np.isclose(ours.intercept_, theirs.intercept_)
assert np.allclose(ours.predict(X_dia), theirs.predict(X_dia))
print("✅ coefficients and predictions match sklearn.linear_model.Ridge")

shared = {k: v for k, v in theirs.get_params().items() if k in ours.get_params()}
assert ours.get_params() == shared
print("✅ get_params matches on shared names:", ours.get_params())

fresh = mini_clone(ours.set_params(alpha=2.0))
assert fresh.get_params() == {"alpha": 2.0, "fit_intercept": True} and not hasattr(fresh, "coef_")
print("✅ mini_clone gives an unfitted copy:", fresh)

sk_cloned = clone(ours)                     # scikit-learn's own clone works on OUR class — duck typing!
print("✅ sklearn.base.clone works too:", sk_cloned, "| fitted?", hasattr(sk_cloned, "coef_"))

before fit → This RidgeRegression instance is not fitted yet. Call 'fit' first.
✅ coefficients and predictions match sklearn.linear_model.Ridge
✅ get_params matches on shared names: {'alpha': 0.5, 'fit_intercept': True}
✅ mini_clone gives an unfitted copy: RidgeRegression(alpha=2.0, fit_intercept=True)
✅ sklearn.base.clone works too: RidgeRegression(alpha=2.0, fit_intercept=True) | fitted? False


### 2) A PyTorch `nn.Module`-style parameter registry

In PyTorch you just write `self.fc1 = nn.Linear(4, 16)` and somehow `model.parameters()` finds every weight. The trick is **`__setattr__`**: `nn.Module` intercepts every attribute assignment and records parameters and sub-modules in dictionaries. `model(x)` works through **`__call__`**, which calls `forward`.

In [36]:
class Parameter:
    def __init__(self, data):
        self.data = np.asarray(data, dtype=np.float32)

    @property
    def shape(self):
        return self.data.shape


class Module:
    def __init__(self):
        object.__setattr__(self, "_parameters", {})     # bypass our own __setattr__ while bootstrapping
        object.__setattr__(self, "_modules", {})

    def __setattr__(self, name, value):
        if isinstance(value, (Parameter, Module)):
            if "_parameters" not in vars(self):
                raise AttributeError("cannot assign parameters or modules before Module.__init__() call")
            registry = self._parameters if isinstance(value, Parameter) else self._modules
            registry[name] = value
        object.__setattr__(self, name, value)

    def named_parameters(self, prefix=""):
        for name, param in self._parameters.items():           # own parameters first…
            yield prefix + name, param
        for name, child in self._modules.items():              # …then each child, with a dotted prefix
            yield from child.named_parameters(prefix + name + ".")

    def parameters(self):
        return [param for _, param in self.named_parameters()]

    def num_parameters(self):
        return sum(param.data.size for param in self.parameters())

    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def forward(self, *args, **kwargs):
        raise NotImplementedError(f"{type(self).__name__} must implement forward()")


class Linear(Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        bound = 1 / np.sqrt(n_in)                               # same init range as torch.nn.Linear
        self.weight = Parameter(rng.uniform(-bound, bound, size=(n_out, n_in)))
        self.bias = Parameter(rng.uniform(-bound, bound, size=n_out))

    def forward(self, x):
        return x @ self.weight.data.T + self.bias.data


class MLP(Module):
    def __init__(self, n_in, hidden, n_out):
        super().__init__()
        self.fc1 = Linear(n_in, hidden)
        self.fc2 = Linear(hidden, n_out)
        self.name = "tiny-mlp"                                  # ordinary attributes are not registered

    def forward(self, x):
        return self.fc2(np.maximum(self.fc1(x), 0))


mlp = MLP(4, 16, 3)
for name, param in mlp.named_parameters():
    print(f"   {name:10s} {param.shape}")
print("total parameters:", mlp.num_parameters())


class Forgetful(Module):
    def __init__(self):
        self.fc = Linear(2, 2)                                  # ❌ forgot super().__init__()


try:
    Forgetful()
except AttributeError as err:
    print("AttributeError:", err)

   fc1.weight (16, 4)
   fc1.bias   (16,)
   fc2.weight (3, 16)
   fc2.bias   (3,)
total parameters: 131
AttributeError: cannot assign parameters or modules before Module.__init__() call


In [37]:
try:
    import torch
except ImportError:
    torch = None
    print("⏭️ Skipped: PyTorch is not installed — `pip install torch` to compare against nn.Module.")

if torch is not None:
    class TorchMLP(torch.nn.Module):
        def __init__(self, n_in, hidden, n_out):
            super().__init__()
            self.fc1 = torch.nn.Linear(n_in, hidden)
            self.fc2 = torch.nn.Linear(hidden, n_out)

        def forward(self, x):
            return self.fc2(torch.relu(self.fc1(x)))

    torch.manual_seed(0)
    torch_mlp = TorchMLP(4, 16, 3)
    ours_named = [(n, p.shape) for n, p in mlp.named_parameters()]
    torch_named = [(n, tuple(p.shape)) for n, p in torch_mlp.named_parameters()]
    assert ours_named == torch_named and mlp.num_parameters() == sum(p.numel() for p in torch_mlp.parameters())
    print("✅ same parameter names, order, shapes, and count as torch.nn.Module")

    for (_, ours_p), (_, torch_p) in zip(mlp.named_parameters(), torch_mlp.named_parameters()):
        ours_p.data = torch_p.detach().numpy().copy()          # load torch's weights into our model

    from sklearn.datasets import load_iris
    x_batch = load_iris().data[:5].astype(np.float32)
    with torch.no_grad():
        torch_out = torch_mlp(torch.from_numpy(x_batch)).numpy()
    assert np.allclose(mlp(x_batch), torch_out, atol=1e-5)
    print("✅ forward pass matches PyTorch on 5 real iris flowers:\n", mlp(x_batch))

✅ same parameter names, order, shapes, and count as torch.nn.Module
✅ forward pass matches PyTorch on 5 real iris flowers:
 [[-0.431  0.978 -0.526]
 [-0.364  0.857 -0.475]
 [-0.41   0.888 -0.481]
 [-0.374  0.823 -0.455]
 [-0.446  0.983 -0.524]]


## ⚠️ Common Pitfalls

### ❌ Pitfall 1 — Mutable default arguments in `__init__`

Default values are created **once**, when the function is defined — not on every call.

In [38]:
class BadDataset:
    def __init__(self, samples=[]):                  # ❌ the same list for every call
        self.samples = samples


d1 = BadDataset()
d1.samples.append("cat.jpg")
print("❌ a brand-new dataset already contains:", BadDataset().samples)


class GoodDataset:
    def __init__(self, samples=None):                # ✅ None as a sentinel
        self.samples = [] if samples is None else list(samples)


d1 = GoodDataset()
d1.samples.append("cat.jpg")
print("✅ a brand-new dataset contains:", GoodDataset().samples)

❌ a brand-new dataset already contains: ['cat.jpg']
✅ a brand-new dataset contains: []


### ❌ Pitfall 2 — Hashing mutable state, then mutating a dict key

In [39]:
class MutablePoint:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))


key = MutablePoint(1, 2)
cache = {key: "embedding for (1, 2)"}
key.x = 99                                            # ❌ hash changes while the object is inside the dict
print("❌ found after mutation:", key in cache, "| but it's still in there:", len(cache))


@dataclass(frozen=True)
class FrozenPoint:
    x: int
    y: int


fkey = FrozenPoint(1, 2)
cache = {fkey: "embedding for (1, 2)"}
try:
    fkey.x = 99                                       # ✅ immutability makes the bug impossible
except FrozenInstanceError as err:
    print("✅ FrozenInstanceError:", err, "| lookup still works:", FrozenPoint(1, 2) in cache)

❌ found after mutation: False | but it's still in there: 1
✅ FrozenInstanceError: cannot assign to field 'x' | lookup still works: True


### ❌ Pitfall 3 — A property that calls itself

In [40]:
class BadScheduler:
    @property
    def lr(self):
        return self.lr                               # ❌ reading self.lr calls this property again… forever


try:
    BadScheduler().lr
except RecursionError as err:
    print("❌ RecursionError:", err)


class GoodScheduler:
    def __init__(self, lr):
        self._lr = lr

    @property
    def lr(self):
        return self._lr                              # ✅ store the value under a different name


print("✅ lr:", GoodScheduler(0.1).lr)

❌ RecursionError: maximum recursion depth exceeded
✅ lr: 0.1


### ❌ Pitfall 4 — Forgetting `super().__init__()`

In [41]:
class Callback:
    def __init__(self):
        self.calls = 0

    def __call__(self, epoch):
        self.calls += 1


class BadPrinter(Callback):
    def __init__(self, prefix):
        self.prefix = prefix                         # ❌ parent never ran, so self.calls doesn't exist


try:
    BadPrinter(">>")(epoch=0)
except AttributeError as err:
    print("❌ AttributeError:", err)


class GoodPrinter(Callback):
    def __init__(self, prefix):
        super().__init__()                           # ✅ let the parent initialise its state first
        self.prefix = prefix


good_printer = GoodPrinter(">>")
good_printer(epoch=0)
print("✅ calls:", good_printer.calls)

❌ AttributeError: 'BadPrinter' object has no attribute 'calls'
✅ calls: 1


### ❌ Pitfall 5 — Putting logic in a scikit-learn estimator's `__init__`

`clone` (used by `GridSearchCV`, `cross_val_score`, …) rebuilds the estimator and checks that every parameter was stored unchanged.

In [42]:
from sklearn.base import BaseEstimator


class BadScaler(BaseEstimator):
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = list(feature_range)     # ❌ modifies the parameter


try:
    clone(BadScaler())
except RuntimeError as err:
    print("❌ RuntimeError:", err)


class GoodScaler(BaseEstimator):
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range           # ✅ store as-is …

    def fit(self, X, y=None):
        low, high = self.feature_range               # … and validate/convert inside fit
        if low >= high:
            raise ValueError("feature_range must be (low, high) with low < high")
        return self


print("✅ clone works:", clone(GoodScaler(feature_range=(-1, 1))))

❌ RuntimeError: Cannot clone object BadScaler(feature_range=[0, 1]), as the constructor either does not set or modifies parameter feature_range
✅ clone works: GoodScaler(feature_range=(-1, 1))


## 🏋️ Practice Exercises

Try each one before opening the solution. Run the cell: ⏳ means not attempted, ✅ means correct.

### 🟢 Exercise 1 — Rectangle with a computed property
Write `Rectangle(width, height)` with a read-only **property** `area` and a `__repr__` that returns exactly `Rectangle(width=3, height=4)`.

In [43]:
Rectangle = None  # TODO
got = None if Rectangle is None else [Rectangle(3, 4).area, repr(Rectangle(3, 4))]
check("rectangle", got, [12, "Rectangle(width=3, height=4)"], hint="Use @property for area and an f-string in __repr__.")

⏳ rectangle: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

    def __repr__(self):
        return f"Rectangle(width={self.width}, height={self.height})"


check("rectangle", [Rectangle(3, 4).area, repr(Rectangle(3, 4))], [12, "Rectangle(width=3, height=4)"])
```
Because `area` is computed on every read, it can never get out of sync with `width` and `height`.
</details>

### 🟢 Exercise 2 — A stack that works with `len()` and `if`
Write `Stack` with `push(item)`, `pop()` (returns the most recent item), `__len__`, and `__bool__` (False when empty).

In [44]:
Stack = None  # TODO
got = None
if Stack is not None:
    s = Stack()
    for item in "abc":
        s.push(item)
    got = [len(s), s.pop(), len(s), bool(Stack()), bool(s)]
check("stack", got, [3, "c", 2, False, True], hint="Keep a list in __init__; pop() can use list.pop().")

⏳ stack: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return len(self._items) > 0


s = Stack()
for item in "abc":
    s.push(item)
check("stack", [len(s), s.pop(), len(s), bool(Stack()), bool(s)], [3, "c", 2, False, True])
```
Without `__bool__`, Python would fall back to `__len__` and give the same result — it's still good to know both.
</details>

### 🟢 Exercise 3 — Fix the shared-state bug
`BuggyAnnotator` shares one label list between all annotators. Write `FixedAnnotator` with the same `add(label)` method so that each annotator keeps its own labels.

In [45]:
class BuggyAnnotator:
    labels = []

    def add(self, label):
        self.labels.append(label)

In [46]:
FixedAnnotator = None  # TODO
got = None
if FixedAnnotator is not None:
    alice, bob = FixedAnnotator(), FixedAnnotator()
    alice.add("cat")
    alice.add("dog")
    bob.add("bird")
    got = [alice.labels, bob.labels]
check("fixed_annotator", got, [["cat", "dog"], ["bird"]], hint="Create self.labels = [] inside __init__.")

⏳ fixed_annotator: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class FixedAnnotator:
    def __init__(self):
        self.labels = []

    def add(self, label):
        self.labels.append(label)


alice, bob = FixedAnnotator(), FixedAnnotator()
alice.add("cat")
alice.add("dog")
bob.add("bird")
check("fixed_annotator", [alice.labels, bob.labels], [["cat", "dog"], ["bird"]])
```
</details>

### 🟡 Exercise 4 — A tokenizer vocabulary
Write `Vocabulary` where:
- `add(token)` gives a **new** token the next id (0, 1, 2, …) and returns it; adding an existing token returns its existing id
- `len(vocab)` is the number of distinct tokens, `vocab[token]` returns the id, and `token in vocab` works

In [47]:
Vocabulary = None  # TODO
got = None
if Vocabulary is not None:
    vocab = Vocabulary()
    got = [vocab.add("the"), vocab.add("cat"), vocab.add("the"), len(vocab), vocab["cat"], "dog" in vocab, "cat" in vocab]
check("vocabulary", got, [0, 1, 0, 2, 1, False, True],
      hint="Store a dict token → id; the next id is len(self._ids). Implement __len__, __getitem__, __contains__.")

⏳ vocabulary: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class Vocabulary:
    def __init__(self):
        self._ids = {}

    def add(self, token):
        if token not in self._ids:
            self._ids[token] = len(self._ids)
        return self._ids[token]

    def __len__(self):
        return len(self._ids)

    def __getitem__(self, token):
        return self._ids[token]

    def __contains__(self, token):
        return token in self._ids


vocab = Vocabulary()
got = [vocab.add("the"), vocab.add("cat"), vocab.add("the"), len(vocab), vocab["cat"], "dog" in vocab, "cat" in vocab]
check("vocabulary", got, [0, 1, 0, 2, 1, False, True])
```
Real tokenizers add the reverse map (id → token) and special tokens such as `<unk>` and `<pad>`.
</details>

### 🟡 Exercise 5 — A plugin registry
Write a decorator factory `register_scaler(name)` that stores the decorated class in `SCALERS[name]` (and returns the class), plus `create_scaler(name, **kwargs)` that instantiates it. Unknown names must raise `ValueError`.

In [48]:
SCALERS = {}
register_scaler = None  # TODO
create_scaler = None    # TODO
got = None
if register_scaler is not None and create_scaler is not None:
    @register_scaler("minmax")
    class MinMax:
        def __init__(self, low=0.0, high=1.0):
            self.low, self.high = low, high

    try:
        create_scaler("robust")
        unknown_raises = False
    except ValueError:
        unknown_raises = True
    scaler = create_scaler("minmax", high=5.0)
    got = [sorted(SCALERS), type(scaler).__name__, scaler.high, MinMax is SCALERS["minmax"], unknown_raises]
check("registry", got, [["minmax"], "MinMax", 5.0, True, True],
      hint="register_scaler(name) returns a decorator(cls) that stores cls and returns it.")

⏳ registry: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
SCALERS = {}


def register_scaler(name):
    def decorator(cls):
        SCALERS[name] = cls
        return cls
    return decorator


def create_scaler(name, **kwargs):
    if name not in SCALERS:
        raise ValueError(f"unknown scaler {name!r}; available: {sorted(SCALERS)}")
    return SCALERS[name](**kwargs)


@register_scaler("minmax")
class MinMax:
    def __init__(self, low=0.0, high=1.0):
        self.low, self.high = low, high


try:
    create_scaler("robust")
    unknown_raises = False
except ValueError:
    unknown_raises = True
scaler = create_scaler("minmax", high=5.0)
check("registry", [sorted(SCALERS), type(scaler).__name__, scaler.high, MinMax is SCALERS["minmax"], unknown_raises],
      [["minmax"], "MinMax", 5.0, True, True])
```
Forgetting `return cls` is the classic bug: the decorated name would become `None`.
</details>

### 🔴 Exercise 6 — Compute the MRO yourself (C3 linearization)

Interviewers at Python-heavy companies sometimes ask you to *explain* C3; implementing it proves you understand it.

Write `c3_mro(hierarchy, name)` where `hierarchy` maps a class name to the **list of its base names** (in order). Return the MRO as a list of names, and raise `TypeError` when no consistent order exists.

**Algorithm:** `L[C] = [C] + merge(L[B1], L[B2], …, [B1, B2, …])`. `merge` repeatedly takes the **first head** (first element of a list) that does **not** appear in the *tail* (everything after the first element) of any list, appends it to the result, and removes it from the front of every list. If no head qualifies, the hierarchy is inconsistent.

The expected answers come from real Python classes built from the same hierarchies:

In [49]:
diamond = {"object": [], "A": ["object"], "B": ["A"], "C": ["A"], "D": ["B", "C"]}
mro_doc_example = {"object": [], "F": ["object"], "E": ["object"], "D": ["object"],
                   "C": ["D", "F"], "B": ["D", "E"], "A": ["B", "C"]}     # the example in Python's MRO document
inconsistent = {"object": [], "X": ["object"], "Y": ["X"], "Z": ["X", "Y"]}


def python_mro(hierarchy, name):
    """Build real classes with type() and read Python's own answer."""
    built = {"object": object}

    def build(n):
        if n not in built:
            built[n] = type(n, tuple(build(b) for b in hierarchy[n]), {})
        return built[n]

    return [k.__name__ for k in build(name).__mro__]


print("Python says D :", python_mro(diamond, "D"))
print("Python says A :", python_mro(mro_doc_example, "A"))

Python says D : ['D', 'B', 'C', 'A', 'object']
Python says A : ['A', 'B', 'C', 'D', 'E', 'F', 'object']


In [50]:
c3_mro = None  # TODO: def c3_mro(hierarchy, name): ...
got = None
if c3_mro is not None:
    try:
        c3_mro(inconsistent, "Z")
        raised = False
    except TypeError:
        raised = True
    got = [c3_mro(diamond, "D"), c3_mro(mro_doc_example, "A"), raised]
check("c3_mro", got, [python_mro(diamond, "D"), python_mro(mro_doc_example, "A"), True],
      hint="Recursively linearize each base, add the list of bases, then merge: pick a head not found in any tail.")

⏳ c3_mro: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def c3_mro(hierarchy, name):
    bases = hierarchy[name]
    sequences = [c3_mro(hierarchy, base) for base in bases] + [list(bases)]
    result = [name]
    while True:
        sequences = [seq for seq in sequences if seq]          # drop exhausted lists
        if not sequences:
            return result
        for seq in sequences:
            head = seq[0]
            if not any(head in other[1:] for other in sequences):
                break                                          # found a good head
        else:
            raise TypeError(f"Cannot create a consistent method resolution order for {name}")
        result.append(head)
        sequences = [seq[1:] if seq[0] == head else seq for seq in sequences]


try:
    c3_mro(inconsistent, "Z")
    raised = False
except TypeError:
    raised = True
got = [c3_mro(diamond, "D"), c3_mro(mro_doc_example, "A"), raised]
check("c3_mro", got, [python_mro(diamond, "D"), python_mro(mro_doc_example, "A"), True])
```

For `Z(X, Y)`: `merge([X, object], [Y, X, object], [X, Y])` — `X` is in the tail of `[Y, X, object]`, and `Y` is in the tail of `[X, Y]`, so no head qualifies → `TypeError`, exactly like Python.
</details>

## 🚀 Mini Project: A scikit-learn-Style Pipeline From Scratch

**Goal:** build `StandardScaler`, `KNNClassifier`, and `MiniPipeline` classes that follow scikit-learn's API, run them on the real **Wine** dataset (178 wines, 13 chemical measurements, 3 grape cultivars), and **prove** they give the same results as scikit-learn.

**Steps:** load data → split train/validation/test → write the classes → verify against scikit-learn → tune `k` on validation (no leakage) → final test evaluation.

### Step 1 — Load the data

In [51]:
from sklearn.datasets import load_wine

wine = load_wine()
X_wine, y_wine = wine.data, wine.target
class_ids, class_counts = np.unique(y_wine, return_counts=True)
print("X:", X_wine.shape, "| wines per class:", dict(zip(class_ids.tolist(), class_counts.tolist())))
col_range = X_wine.max(axis=0) - X_wine.min(axis=0)
print(f"feature ranges differ hugely: '{wine.feature_names[int(col_range.argmax())]}' spans {col_range.max():.0f}, "
      f"'{wine.feature_names[int(col_range.argmin())]}' spans {col_range.min():.2f} → distances need scaling")

X: (178, 13) | wines per class: {0: 59, 1: 71, 2: 48}
feature ranges differ hugely: 'proline' spans 1402, 'nonflavanoid_phenols' spans 0.53 → distances need scaling


### Step 2 — Split 60 / 20 / 20 (train / validation / test)

We choose `k` on the **validation** set and touch the **test** set exactly once at the end.

In [52]:
from sklearn.model_selection import train_test_split

X_trval, X_test, y_trval, y_test = train_test_split(X_wine, y_wine, test_size=0.2, stratify=y_wine, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, stratify=y_trval, random_state=42)
print("train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)

train: (106, 13) | val: (36, 13) | test: (36, 13)


### Step 3 — Write the classes (reusing `MiniBaseEstimator` and `mini_clone` from 🔧 Build It From Scratch)

In [53]:
class StandardScaler(MiniBaseEstimator):
    def __init__(self, with_mean=True, with_std=True):
        self.with_mean = with_mean
        self.with_std = with_std

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.mean_ = X.mean(axis=0) if self.with_mean else np.zeros(X.shape[1])
        std = X.std(axis=0) if self.with_std else np.ones(X.shape[1])
        self.scale_ = np.where(std == 0, 1.0, std)             # constant columns: don't divide by zero
        return self

    def transform(self, X):
        self._check_is_fitted()
        X = np.asarray(X, dtype=float)
        if X.shape[1] != self.n_features_in_:
            raise ValueError(f"expected {self.n_features_in_} features, got {X.shape[1]}")
        return (X - self.mean_) / self.scale_

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


class KNNClassifier(MiniBaseEstimator):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        if self.n_neighbors < 1:                                # validate in fit, not __init__
            raise ValueError("n_neighbors must be >= 1")
        self.X_train_ = np.asarray(X, dtype=float)
        self.classes_, self.y_encoded_ = np.unique(np.asarray(y), return_inverse=True)
        return self

    def predict_proba(self, X):
        self._check_is_fitted()
        X = np.asarray(X, dtype=float)
        dist = ((X[:, None, :] - self.X_train_[None, :, :]) ** 2).sum(axis=2)   # (n_query, n_train)
        nearest = np.argsort(dist, axis=1, kind="stable")[:, :self.n_neighbors]
        votes = self.y_encoded_[nearest]                                        # (n_query, k)
        counts = np.stack([(votes == c).sum(axis=1) for c in range(len(self.classes_))], axis=1)
        return counts / self.n_neighbors

    def predict(self, X):
        proba = self.predict_proba(X)                                          # runs the fitted check first
        return self.classes_[proba.argmax(axis=1)]                             # ties → lowest class label

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


class MiniPipeline(MiniBaseEstimator):
    def __init__(self, steps):
        self.steps = steps

    def get_params(self, deep=True):
        params = {"steps": self.steps}
        if deep:                                               # expose nested params as "step__param"
            for step_name, step in self.steps:
                params[step_name] = step
                for key, value in step.get_params(deep=False).items():
                    params[f"{step_name}__{key}"] = value
        return params

    def set_params(self, **params):
        named = dict(self.steps)
        for key, value in params.items():
            step_name, sep, param = key.partition("__")
            if not sep or step_name not in named:
                raise ValueError(f"use 'step__param' names; steps are {list(named)}, got {key!r}")
            named[step_name].set_params(**{param: value})
        return self

    def __repr__(self):
        return f"MiniPipeline({[(n, s) for n, s in self.steps]})"

    def fit(self, X, y):
        for _, step in self.steps[:-1]:
            X = step.fit_transform(X, y)
        self.steps[-1][1].fit(X, y)
        return self

    def _transform(self, X):
        for _, step in self.steps[:-1]:
            X = step.transform(X)
        return X

    def predict(self, X):
        return self.steps[-1][1].predict(self._transform(X))

    def predict_proba(self, X):
        return self.steps[-1][1].predict_proba(self._transform(X))

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


pipe = MiniPipeline([("scaler", StandardScaler()), ("knn", KNNClassifier(n_neighbors=5))])
print(pipe)
print("nested params:", {k: v for k, v in pipe.get_params().items() if "__" in k})

MiniPipeline([('scaler', StandardScaler(with_mean=True, with_std=True)), ('knn', KNNClassifier(n_neighbors=5))])
nested params: {'scaler__with_mean': True, 'scaler__with_std': True, 'knn__n_neighbors': 5}


### Step 4 — Verify against scikit-learn

In [54]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler as SkStandardScaler

our_scaler = StandardScaler().fit(X_train)
sk_scaler = SkStandardScaler().fit(X_train)
assert np.allclose(our_scaler.mean_, sk_scaler.mean_) and np.allclose(our_scaler.scale_, sk_scaler.scale_)
assert np.allclose(our_scaler.transform(X_val), sk_scaler.transform(X_val))
print("✅ StandardScaler: mean_, scale_ and transform match scikit-learn")

for k in (1, 5, 15):
    ours_pipe = MiniPipeline([("scaler", StandardScaler()), ("knn", KNNClassifier(n_neighbors=k))]).fit(X_train, y_train)
    sk_pipe = Pipeline([("scaler", SkStandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))]).fit(X_train, y_train)
    assert np.array_equal(ours_pipe.predict(X_val), sk_pipe.predict(X_val))
    assert np.allclose(ours_pipe.predict_proba(X_val), sk_pipe.predict_proba(X_val))
    print(f"✅ k={k:2d}: identical predictions and probabilities | val accuracy {ours_pipe.score(X_val, y_val):.3f}")

pipe.set_params(knn__n_neighbors=7)
print("set_params routed into the step:", pipe.steps[1][1])
try:
    KNNClassifier().predict(X_val)
except RuntimeError as err:
    print("unfitted predict →", err)

✅ StandardScaler: mean_, scale_ and transform match scikit-learn
✅ k= 1: identical predictions and probabilities | val accuracy 0.944
✅ k= 5: identical predictions and probabilities | val accuracy 0.972
✅ k=15: identical predictions and probabilities | val accuracy 1.000
set_params routed into the step: KNNClassifier(n_neighbors=7)
unfitted predict → This KNNClassifier instance is not fitted yet. Call 'fit' first.


### Step 5 — Tune `k` on the validation set (and measure what scaling is worth)

In [55]:
template = MiniPipeline([("scaler", StandardScaler()), ("knn", KNNClassifier())])
no_scaling = MiniPipeline([("knn", KNNClassifier())])

rows = []
for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    scaled_acc = mini_clone(template).set_params(knn__n_neighbors=k).fit(X_train, y_train).score(X_val, y_val)
    raw_acc = mini_clone(no_scaling).set_params(knn__n_neighbors=k).fit(X_train, y_train).score(X_val, y_val)
    rows.append((k, scaled_acc, raw_acc))
    print(f"k={k:2d} | val accuracy with scaling {scaled_acc:.3f} | without scaling {raw_acc:.3f}")

best_k, best_val, _ = max(rows, key=lambda r: (r[1], -r[0]))     # highest accuracy, then smaller k
mean_gain = np.mean([s - r for _, s, r in rows])
print(f"\nbest k on validation: {best_k} (accuracy {best_val:.3f})")
print(f"scaling changes validation accuracy by {mean_gain:+.3f} on average across k")
assert template.steps[1][1].get_params() == {"n_neighbors": 5}, "mini_clone must not modify the template"

k= 1 | val accuracy with scaling 0.944 | without scaling 0.722
k= 3 | val accuracy with scaling 0.972 | without scaling 0.722
k= 5 | val accuracy with scaling 0.972 | without scaling 0.694
k= 7 | val accuracy with scaling 1.000 | without scaling 0.667
k= 9 | val accuracy with scaling 1.000 | without scaling 0.639
k=11 | val accuracy with scaling 1.000 | without scaling 0.639
k=15 | val accuracy with scaling 1.000 | without scaling 0.667
k=21 | val accuracy with scaling 1.000 | without scaling 0.639

best k on validation: 7 (accuracy 1.000)
scaling changes validation accuracy by +0.313 on average across k


### Step 6 — Final evaluation on the untouched test set

In [56]:
final = mini_clone(template).set_params(knn__n_neighbors=best_k).fit(X_trval, y_trval)
sk_final = Pipeline([("scaler", SkStandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=best_k))]).fit(X_trval, y_trval)

test_pred = final.predict(X_test)
assert np.array_equal(test_pred, sk_final.predict(X_test))
test_acc = float(np.mean(test_pred == y_test))
majority_acc = float(np.mean(y_test == np.bincount(y_trval).argmax()))
print(f"test accuracy (k={best_k}, refit on train+val): {test_acc:.3f} | scikit-learn: {sk_final.score(X_test, y_test):.3f}")
print(f"majority-class baseline: {majority_acc:.3f} → our pipeline is {test_acc - majority_acc:+.3f} better")

confusion = np.zeros((3, 3), dtype=int)
np.add.at(confusion, (y_test, test_pred), 1)
print("confusion matrix (rows = true cultivar, cols = predicted):\n", confusion)

test accuracy (k=7, refit on train+val): 1.000 | scikit-learn: 1.000


majority-class baseline: 0.389 → our pipeline is +0.611 better
confusion matrix (rows = true cultivar, cols = predicted):
 [[12  0  0]
 [ 0 14  0]
 [ 0  0 10]]


**Stretch goals**
1. Add `weights="distance"` to `KNNClassifier` (closer neighbours get bigger votes) and verify against scikit-learn.
2. Write a `cross_val_score(estimator, X, y, cv=5)` function that uses `mini_clone` for each fold.
3. Make `KNNClassifier` work with scikit-learn's own `cross_val_score` and `GridSearchCV` by inheriting `ClassifierMixin, BaseEstimator` (in that order — why?).

### 🗣️ How to talk about this in an interview
- "I rebuilt scikit-learn's estimator contract: hyperparameters only stored in `__init__`, learned state in trailing-underscore attributes, `fit` returns `self`, and `get_params`/`set_params` derived from the `__init__` signature, which is what makes `clone` and grid search work."
- "My `MiniPipeline` composes steps instead of inheriting from them and routes nested parameters with the `step__param` convention."
- "I verified the scaler statistics and every k-NN prediction and probability against scikit-learn with asserts, including tie-breaking, which needed a stable sort."
- "I tuned `k` on a validation split, refit on train+validation, and touched the test set once; I also showed that scaling matters for distance-based models."

## 🎤 Interview Q&A

Try answering **out loud** before opening each answer.

### 🧠 Concepts

**Q1. What is the MRO, and what does `super()` really call in multiple inheritance?**

<details><summary>Show answer</summary>

- **30-second answer:** The method resolution order is the linear list of classes Python searches for an attribute, computed with C3 linearization: children before parents, parents in the listed order, and each parent's own order preserved. `super()` returns the **next class after the current one in the MRO of `type(self)`**, so in `class D(B, C)` a `super()` call inside `B` reaches `C`, not `A`.
- **Go deeper:** Inspect it with `D.__mro__` or `D.mro()`. Cooperative multiple inheritance requires every class to call `super()` (and pass `**kwargs` through) so each `__init__` runs exactly once. If no consistent order exists, class creation fails with `TypeError: Cannot create a consistent method resolution order`.
- **❌ Common wrong answer:** "`super()` calls the parent class" or "Python searches depth-first, left to right" (that was old-style classes in Python 2).

</details>

**Q2. How are `__eq__` and `__hash__` related?**

<details><summary>Show answer</summary>

- **30-second answer:** Objects that compare equal must have equal hashes, because sets and dicts find items by hash first and then by `==`. If you define `__eq__` without `__hash__`, Python sets `__hash__ = None`, so the objects become unhashable. Hash the same immutable fields you compare, or use `@dataclass(frozen=True)`.
- **Go deeper:** The hash must not change while the object is inside a set/dict, so don't hash mutable state. Return `NotImplemented` from `__eq__` for unrelated types. A plain `@dataclass` is unhashable; `unsafe_hash=True` forces a hash.
- **❌ Common wrong answer:** "Hash just needs to be unique, so I'll use `id(self)` while comparing by value" — then equal objects land in different buckets and deduplication silently fails.

</details>

**Q3. Composition vs inheritance — how do you choose?**

<details><summary>Show answer</summary>

- **30-second answer:** Inheritance models "is-a" and couples the child to the parent's implementation; composition models "has-a" and combines small objects passed in. Prefer composition when behaviours vary independently (optimizer, logger, callbacks, preprocessing steps); use inheritance for true subtypes and framework extension points like `nn.Module` or `BaseEstimator`.
- **Go deeper:** Inheritance for every combination explodes (models × loggers × stoppers subclasses). Composition enables dependency injection, so tests can pass fakes. scikit-learn's `Pipeline` is composition. Use the Liskov substitution principle as the test for inheritance.
- **❌ Common wrong answer:** "Inheritance is always the OOP way to reuse code."

</details>

**Q4. Abstract base class or `typing.Protocol`?**

<details><summary>Show answer</summary>

- **30-second answer:** An ABC is nominal: classes must inherit from it, and Python refuses to instantiate a subclass that misses an abstract method. A Protocol is structural: any class with matching methods satisfies it, checked by static type checkers such as mypy or pyright, no inheritance needed. Use an ABC when you own the hierarchy and want enforcement plus shared methods; use a Protocol to accept third-party objects.
- **Go deeper:** `@runtime_checkable` enables `isinstance`, but it only checks that attribute names exist, not signatures. ABCs can supply mixin methods (e.g. `collections.abc.Sequence` gives `__contains__`, `__iter__`, `index`, `count`). `SomeABC.register(cls)` creates virtual subclasses.
- **❌ Common wrong answer:** "Protocols are enforced at runtime just like ABCs."

</details>

**Q5. When would you use a dataclass, a NamedTuple, a regular class, or a pydantic model?**

<details><summary>Show answer</summary>

- **30-second answer:** `NamedTuple` for small immutable records that unpack like tuples; `@dataclass` for configs and internal records with defaults (optionally `frozen`, `slots`, `kw_only`); a regular class when behaviour and invariants dominate; pydantic when data comes from outside (JSON, HTTP requests) and must be parsed and validated.
- **Go deeper:** A namedtuple *is* a tuple, so `Point(1, 2) == (1, 2)` is `True` — sometimes surprising. Dataclasses don't check annotation types at runtime. Mutable defaults need `field(default_factory=...)`.
- **❌ Common wrong answer:** "Dataclasses validate the types in the annotations."

</details>

**Q6. `@classmethod` vs `@staticmethod` — and why are class methods used as factories?**

<details><summary>Show answer</summary>

- **30-second answer:** A class method receives the class as `cls`, so `cls(...)` builds an instance of whichever class it was called on — perfect for alternative constructors like `from_json` or `from_pretrained` that keep working for subclasses. A static method receives nothing; it's a function stored in the class namespace.
- **Go deeper:** Standard-library examples: `dict.fromkeys`, `datetime.fromisoformat`, `int.from_bytes`. Hard-coding the class name inside a factory breaks subclassing. A module-level function is often clearer than a static method.
- **❌ Common wrong answer:** "Class methods can read instance attributes" or "they're the same as static methods."

</details>

**Q7. What does `__slots__` do, and when is it worth using?**

<details><summary>Show answer</summary>

- **30-second answer:** It declares a fixed set of attribute names so instances don't get a per-object `__dict__`: less memory and slightly faster attribute access, but you can't add new attributes. Worth it for millions of small objects.
- **Go deeper:** Measure with `tracemalloc` (`sys.getsizeof` hides the dict). Since Python 3.11 regular instance dicts are more compact, so the saving is smaller than it used to be. Every class in the hierarchy needs `__slots__`; add `"__weakref__"` for weak references; `@dataclass(slots=True)` is the shortcut.
- **❌ Common wrong answer:** "`__slots__` makes attributes private or immutable."

</details>

**Q8. What is the difference between `__repr__` and `__str__`?**

<details><summary>Show answer</summary>

- **30-second answer:** `__repr__` is the unambiguous developer view (REPL, logs, debuggers), ideally looking like code that recreates the object; `__str__` is the friendly end-user view used by `print` and f-strings. If only `__repr__` exists, `str()` falls back to it — so always write `__repr__` first.
- **Go deeper:** Containers show their elements with `repr`, so `print([obj])` uses `__repr__`. `f"{x!r}"` forces repr. scikit-learn estimators show their hyperparameters in `__repr__`.
- **❌ Common wrong answer:** "Printing a list calls `__str__` on each item."

</details>

### 💻 Coding

**Q9. Design a model registry so new models can be added without editing the factory function.**

<details><summary>Show answer</summary>

- **30-second answer:** Keep a dict from name to class; a decorator `@register("name")` adds the class and returns it unchanged; `create(name, **kwargs)` looks it up and raises a helpful error listing valid names. Alternatively, `__init_subclass__` auto-registers every subclass.
- **Go deeper:** Reject duplicate names. Classes only register once their module is imported — import model modules in the package `__init__` or use packaging entry points for plugins from other packages. This is the open/closed principle.
- **❌ Common wrong answer:** A growing `if name == ...: elif ...` chain that must be edited for every new model.

</details>

**Q10. Write a dataset class that works with `len()`, indexing, and `for` loops (like a PyTorch `Dataset`).**

<details><summary>Show answer</summary>

- **30-second answer:** Implement `__len__` and `__getitem__(i)` (a *map-style* dataset); raise `IndexError` for out-of-range indices so plain iteration stops. Or implement `__iter__` for a stream (*iterable-style*).
- **Go deeper:** `torch.utils.data.DataLoader` calls `__getitem__` with indices chosen by a sampler, which uses `__len__`. Load data lazily inside `__getitem__` for big datasets. An iterable's `__iter__` should return a *new* iterator each time so you can loop twice.
- **❌ Common wrong answer:** "You have to inherit from `list`."

</details>

**Q11. What rules must a scikit-learn-compatible estimator follow?**

<details><summary>Show answer</summary>

- **30-second answer:** `__init__` only stores hyperparameters as same-named attributes (no validation or computation); `fit` validates input, learns, stores results in attributes ending in `_`, and returns `self`; provide `predict`/`transform`; inherit `BaseEstimator` for `get_params`/`set_params`/repr/tags, with mixins such as `ClassifierMixin` on the left.
- **Go deeper:** `clone` rebuilds estimators from `get_params()` and raises `RuntimeError` if `__init__` altered a parameter; `GridSearchCV` uses `set_params`, bypassing `__init__`. `check_is_fitted` looks for trailing-underscore attributes. `sklearn.utils.estimator_checks.check_estimator` tests compliance; recent versions use `__sklearn_tags__`, which `BaseEstimator` provides.
- **❌ Common wrong answer:** "Validate hyperparameters in `__init__` so errors show up early."

</details>

### 🐛 Debugging Scenarios

**Q12. Every `Experiment` object shows the metrics of *all* experiments. What's wrong?**

<details><summary>Show answer</summary>

- **30-second answer:** The metrics list is shared: either a class attribute (`metrics = []` in the class body) or a mutable default argument (`def __init__(self, metrics=[])`). Create a new list per object in `__init__`, using `None` as the default.
- **Go deeper:** Confirm with `a.metrics is b.metrics`. Default values are evaluated once, when the `def` runs. Ruff flags these: `B006` (mutable argument default) and `RUF012` (mutable class attribute).
- **❌ Common wrong answer:** "Python copies lists for each instance automatically."

</details>

**Q13. An object you added to a set is no longer found with `in`. Why?**

<details><summary>Show answer</summary>

- **30-second answer:** Its `__hash__` depends on fields that were mutated after insertion, so the lookup searches the wrong bucket. Only hash immutable state — use a frozen dataclass or tuple — or never mutate objects used as keys.
- **Go deeper:** `x in s` computes `hash(x)`, then compares with `==` inside that bucket. The same applies to dict keys. Rebuilding the set after mutation is a workaround, not a fix.
- **❌ Common wrong answer:** "Sets are unordered, so sometimes items get lost."

</details>

**Q14. Your PyTorch model raises `AttributeError: cannot assign module before Module.__init__() call`. What happened?**

<details><summary>Show answer</summary>

- **30-second answer:** The subclass's `__init__` assigned a layer before calling `super().__init__()`. `nn.Module.__setattr__` needs the internal parameter/module dictionaries that the parent's `__init__` creates. Call `super().__init__()` first.
- **Go deeper:** The same class of bug gives a plain `AttributeError` in your own classes when a parent's attributes were never set. With dataclasses, remember to call `super().__post_init__()` if the parent defines one.
- **❌ Common wrong answer:** "Reinstall PyTorch" or "the layer sizes are wrong."

</details>

### 🏗️ Design

**Q15. How do SOLID principles show up in a real ML training code base?**

<details><summary>Show answer</summary>

- **30-second answer:** **S**ingle responsibility: trainer loops, model computes, logger logs, metric scores. **O**pen/closed: a registry plus a small interface lets you add models without editing the trainer. **L**iskov: any model subclass works wherever the base is expected. **I**nterface segregation: small protocols like `SupportsPredict` instead of one giant base class. **D**ependency inversion: the trainer receives logger and metric objects (dependency injection), so tests can pass fakes.
- **Go deeper:** Extension hooks (Keras callbacks, Lightning hooks) are the template method pattern. Config-driven construction goes through the registry. Don't over-engineer: start with functions and extract classes when a second implementation actually appears.
- **❌ Common wrong answer:** Reciting the acronym with no example, or proposing a deep hierarchy like `BaseModel → TorchModel → TorchClassifier → TorchImageClassifier`.

</details>

## 🧪 Quick Quiz

Predict the output, then reveal.

**1.**
```python
class Model:
    history = []
a, b = Model(), Model()
a.history.append("run1")
print(b.history)
```
<details><summary>Answer</summary>

`['run1']` — `history` is a class attribute, so both objects share one list.
</details>

**2.** With `class B(A)`, `class C(A)`, `class D(B, C)`, what is `[k.__name__ for k in D.__mro__]`?
<details><summary>Answer</summary>

`['D', 'B', 'C', 'A', 'object']` — C3 puts `A` after *both* of its children.
</details>

**3.**
```python
@dataclass
class P:
    x: int
print(P(1) == P(1))
{P(1)}
```
<details><summary>Answer</summary>

Prints `True`, then raises `TypeError: unhashable type: 'P'` — the generated `__eq__` sets `__hash__` to `None`.
</details>

**4.**
```python
class Box:
    def __len__(self):
        return 0
print(bool(Box()))
```
<details><summary>Answer</summary>

`False` — without `__bool__`, truthiness falls back to `__len__`.
</details>

**5.**
```python
class A:
    def __init__(self): print("A", end=" ")
class B(A):
    def __init__(self): print("B", end=" ")
B()
```
<details><summary>Answer</summary>

Only `B` — overriding `__init__` without calling `super().__init__()` means `A.__init__` never runs.
</details>

## 📚 Resources

### 📖 Official Docs
- [9. Classes — Python tutorial](https://docs.python.org/3/tutorial/classes.html) — the official introduction to classes, inheritance, and iterators
- [3. Data model](https://docs.python.org/3/reference/datamodel.html) — the complete list of dunder methods and what triggers them
- [dataclasses — Data Classes](https://docs.python.org/3/library/dataclasses.html) · [abc — Abstract Base Classes](https://docs.python.org/3/library/abc.html)
- [Protocols — typing documentation](https://typing.python.org/en/latest/spec/protocol.html) — structural subtyping, as type checkers implement it
- [The Python 2.3 Method Resolution Order](https://docs.python.org/3/howto/mro.html) — the C3 algorithm explained with worked examples (still how Python 3 works)
- [Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html) — the official estimator contract we rebuilt
- [Module — PyTorch 2.14 documentation](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Module.html) — the real `nn.Module` API

### 🎥 Videos
- [Corey Schafer — Python OOP Tutorials - Working with Classes](https://www.youtube.com/playlist?list=PL-osiE80TeTsqhIuOqKhwlXsIBIdSeYtc) (playlist of short videos) — the clearest beginner walkthrough of classes, inheritance, dunders, and properties
- [Raymond Hettinger — Python's Class Development Toolkit](https://www.youtube.com/watch?v=HTLu2DFOdTg) (~46 min) — a Python core developer shows *why* each class feature (class methods, properties, `__slots__`) exists, through one evolving example
- [ArjanCodes — Protocols vs ABCs in Python - When to Use Which One?](https://www.youtube.com/watch?v=dryNwWvSd4M) (~16 min) — practical comparison of nominal vs structural interfaces

### 📄 Papers & Specifications
- [PEP 544 – Protocols: Structural subtyping (static duck typing)](https://peps.python.org/pep-0544/)
- [PEP 557 – Data Classes](https://peps.python.org/pep-0557/)
- [PEP 487 – Simpler customisation of class creation](https://peps.python.org/pep-0487/) — introduced `__init_subclass__`

### 📘 Books & Courses
- [Python’s super() considered super! — Raymond Hettinger](https://rhettinger.wordpress.com/2011/05/26/super-considered-super/) — the classic article on cooperative multiple inheritance
- [A Brief Interlude: On Coupling and Abstractions](https://www.cosmicpython.com/book/chapter_03_abstractions.html) — a free chapter of *Architecture Patterns with Python* (Percival & Gregory) on composition, dependency injection, and testable design
- [Object-Oriented Programming (OOP) in Python – Real Python](https://realpython.com/python3-object-oriented-programming/) — a beginner-friendly second explanation

### 🏋️ Practice
- [Design Patterns in Python](https://refactoring.guru/design-patterns/python) — every classic pattern with Python code, useful for design-round vocabulary
- Re-implement one more scikit-learn class (e.g. `MinMaxScaler` or `LogisticRegression`) with the `MiniBaseEstimator` from this notebook and assert it matches.

## 📝 Summary Cheat Sheet

| Concept | What it does | Key API / rule |
|---|---|---|
| Class / object | blueprint / thing built from it | `class C:` · `obj = C()` · `self` = the object |
| Attributes | data on objects or shared by the class | `self.x` per object · class body = shared (constants only!) |
| Methods | behaviour | instance (`self`) · `@classmethod` (`cls`, factories) · `@staticmethod` |
| Property | attribute access that runs code | `@property` + `@x.setter`; store in `self._x` |
| Inheritance | is-a reuse | `class Child(Parent)` · `super().__init__()` first |
| MRO | attribute search order | C3 linearization · `Cls.__mro__` · `super()` = next in MRO |
| Composition | has-a reuse | inject collaborators in `__init__` |
| ABC | nominal contract, enforced at instantiation | `ABC`, `@abstractmethod`, `collections.abc` |
| Protocol | structural contract for type checkers | `typing.Protocol`, `@runtime_checkable` (names only) |
| Dunders | plug into Python syntax | `__repr__`, `__eq__`+`__hash__`, `__len__`, `__getitem__`, `__iter__`, `__call__`, `NotImplemented` |
| Dataclass | generated boilerplate | `@dataclass(frozen, order, slots)`, `field(default_factory=…)`, `replace`, `asdict` |
| `__slots__` | no per-object dict, less memory | `__slots__ = ("x", "y")` — measure with `tracemalloc` |
| Patterns | strategy · registry · template method | functions as strategies · decorator / `__init_subclass__` registry · hook methods |
| sklearn API | estimator contract | params in `__init__` · `fit` → `self` · learned `attr_` · `get_params`/`set_params`/`clone` |
| nn.Module idea | automatic parameter tracking | `__setattr__` registry · `named_parameters()` · `__call__` → `forward` |

## ➡️ What's Next

**[04 · Virtual Environments & Packaging](04_Virtual_Env_and_Packaging.ipynb)** — now that you can write reusable classes, learn how to put them in a proper package with `pyproject.toml`, isolate dependencies with `uv`, and install and test your code like a real library.